[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab4/Lab4_Transformer_MagnaTagATune.ipynb)

# Lab 4 — Transformer Encoder for MagnaTagATune

This notebook adapts the class 8 Transformer to the MagnaTagATune (MTAT) dataset, following the four steps of the reference notebook:

1. Load and preprocess the dataset
2. Define the Transformer model
3. Train the model
4. Evaluate the model

**Objective:** train a Transformer Encoder for multi-label classification and evaluate its performance using ROC-AUC and PR-AUC.

The notebook starts from the in-class architecture and incorporates the adaptations required for MTAT. It includes two controlled experiments, each changing a single parameter relative to the baseline:
- Experiment 1: increase `emb_dim` from 64 to 128.
- Experiment 2: increase `n_blocks` from 2 to 4.

Results and conclusions should be completed after running the notebook. No improvement over the baseline is assumed in advance.

## 1. Load and preprocess the dataset

### What this section does

MTAT audio files and labels are loaded using the official splits `train_gt_mtt.tsv`, `val_gt_mtt.tsv`, and `test_gt_mtt.tsv`. Each example retains a multi-hot label vector: a track can have multiple positive labels.

### Data loading strategy

Audio files remain on the local Colab disk (`/content/mtat/mp3/`). The DataLoader reads and preprocesses only the files needed for the current batches. The complete audio dataset is never stored in RAM.

- The subset of genre tags defined in `GENRES_TAGS`.
- Only clips with at least one of those tags are retained.
- Audio is converted to mono and resampled to 16 kHz.
- A random 5-second segment is cropped; if the available audio is shorter, it is zero-padded.
- Batch workers load audio from disk and prepare batches in parallel.

The initial lab configuration uses 5-second segments and the full dataset.

In [ ]:
# !!! MODIFICATION: Install the dependencies required to run the notebook with MTAT.

# Install dependencies in Colab if needed.
# This cell can be skipped if the dependencies are already installed.
%pip install -q torch torchaudio scikit-learn pandas matplotlib tqdm

## 0. Download MagnaTagATune

This cell downloads the annotations, the official train/validation/test splits, and the audio files. Downloads can be resumed if interrupted. The directory structure is shared with Lab 3 (`/content/mtat`).

Run this cell once before loading the datasets. If the files are already complete, existing downloads and extracted audio folders are reused.

In [ ]:
# Download MagnaTagATune annotations, official splits, and audio files.
# Downloads are resumable and checked against the server-reported file size.

import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Keep the same directory structure used in Lab 3.
MTAT_ROOT = Path("/content/mtat")
AUDIO_DIR = MTAT_ROOT / "mp3"
ANNOTATIONS_PATH = MTAT_ROOT / "annotations_final.csv"

MTAT_ROOT.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def download_with_resume(url, destination):
    """Download a file with wget resume support and verify its size."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    request = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(request, timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length", 0))

    downloaded_path = destination.parent / os.path.basename(url)

    if downloaded_path.exists() and expected_size:
        if downloaded_path.stat().st_size > expected_size:
            downloaded_path.unlink()

    print(f"Downloading {downloaded_path.name}...")
    subprocess.run(
        [
            "wget", "-c", "--tries=5", "--timeout=30",
            url, "-P", str(destination.parent)
        ],
        check=True
    )

    if not downloaded_path.exists():
        raise FileNotFoundError(f"Download did not create: {downloaded_path}")

    actual_size = downloaded_path.stat().st_size
    if actual_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {downloaded_path}")

    if expected_size and actual_size != expected_size:
        raise RuntimeError(
            f"Incomplete download for {downloaded_path.name}: "
            f"{actual_size:,} bytes downloaded, expected {expected_size:,}. "
            "Run this cell again to resume the download."
        )

    print(f"Verified: {downloaded_path.name} ({actual_size:,} bytes)")
    return downloaded_path


# Download annotations from the official MagnaTagATune host.
download_with_resume(
    "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv",
    ANNOTATIONS_PATH
)


# Download the official Musicnn train, validation, and test split files.
SPLIT_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "jordipons/musicnn-training/master/data/index/mtt/"
)

for split_name in [
    "train_gt_mtt.tsv",
    "val_gt_mtt.tsv",
    "test_gt_mtt.tsv",
]:
    download_with_resume(
        SPLIT_BASE_URL + split_name,
        MTAT_ROOT / split_name
    )


# The audio archive is distributed in three parts and contains 16 shards (0-f).
expected_shards = set("0123456789abcdef")

# Move shards into AUDIO_DIR if a previous extraction placed them one level too high.
for shard in expected_shards:
    old_location = MTAT_ROOT / shard
    new_location = AUDIO_DIR / shard
    if old_location.is_dir() and not new_location.exists():
        shutil.move(str(old_location), str(new_location))


def audio_shards_are_complete():
    """Check that all 16 shard folders exist and contain MP3 files."""
    for shard in expected_shards:
        shard_dir = AUDIO_DIR / shard
        if not shard_dir.is_dir() or not any(shard_dir.rglob("*.mp3")):
            return False
    return True


if not audio_shards_are_complete():
    audio_base_url = "https://mirg.city.ac.uk/datasets/magnatagatune/"

    archive_parts = [
        "mp3.zip.001",
        "mp3.zip.002",
        "mp3.zip.003",
    ]

    for part in archive_parts:
        download_with_resume(
            audio_base_url + part,
            MTAT_ROOT / part
        )

    # Rebuild the original ZIP archive from its downloaded parts.
    archive_path = MTAT_ROOT / "mp3.zip"
    print("Joining the three archive parts...")

    with archive_path.open("wb") as output_file:
        for part in archive_parts:
            with (MTAT_ROOT / part).open("rb") as input_file:
                shutil.copyfileobj(input_file, output_file)

    # Test the reconstructed archive before extracting it.
    subprocess.run(
        ["unzip", "-tq", str(archive_path)],
        check=True
    )

    print("Extracting audio files...")
    subprocess.run(
        ["unzip", "-q", "-o", str(archive_path), "-d", str(AUDIO_DIR)],
        check=True
    )

    # Handle archives that extract the hexadecimal folders one level too high.
    for shard in expected_shards:
        old_location = AUDIO_DIR / shard
        new_location = AUDIO_DIR / shard
        root_location = MTAT_ROOT / shard

        if not old_location.exists() and root_location.is_dir():
            shutil.move(str(root_location), str(new_location))

    missing_shards = [
        shard for shard in sorted(expected_shards)
        if not (AUDIO_DIR / shard).is_dir()
        or not any((AUDIO_DIR / shard).rglob("*.mp3"))
    ]
    if missing_shards:
        raise RuntimeError(
            f"Audio extraction is incomplete. Missing or empty shards: {missing_shards}"
        )
else:
    print("All 16 audio folders are already extracted.")


# Final checks before the dataset is loaded.
required_files = [
    ANNOTATIONS_PATH,
    MTAT_ROOT / "train_gt_mtt.tsv",
    MTAT_ROOT / "val_gt_mtt.tsv",
    MTAT_ROOT / "test_gt_mtt.tsv",
]

for file_path in required_files:
    if not file_path.is_file() or file_path.stat().st_size == 0:
        raise RuntimeError(f"Required file is missing or empty: {file_path}")

audio_count = sum(1 for _ in AUDIO_DIR.rglob("*.mp3"))

print("\nMagnaTagATune download completed.")
print(f"Dataset directory: {MTAT_ROOT}")
print(f"Annotations: {ANNOTATIONS_PATH.stat().st_size:,} bytes")
print(f"Train split: {(MTAT_ROOT / 'train_gt_mtt.tsv').stat().st_size:,} bytes")
print(f"Validation split: {(MTAT_ROOT / 'val_gt_mtt.tsv').stat().st_size:,} bytes")
print(f"Test split: {(MTAT_ROOT / 'test_gt_mtt.tsv').stat().st_size:,} bytes")
print(f"Audio files found: {audio_count:,}")
print(f"Audio folders: {sorted(expected_shards)}")


In [ ]:
# !!! MODIFICATION: Adapted imports: add pandas and multi-label metrics; automatically select CPU/GPU instead of forcing CUDA.

import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from torchaudio.transforms import MelSpectrogram, Resample
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm, trange

# Basic reproducibility. Audio cropping also depends on these random seeds.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Expected paths:
# /content/mtat/annotations_final.csv
# /content/mtat/train_gt_mtt.tsv
# /content/mtat/val_gt_mtt.tsv
# /content/mtat/test_gt_mtt.tsv
# /content/mtat/mp3/<mp3 files>

DATA_DIR = Path("/content/mtat")
ANNOTATION_FILE = DATA_DIR / "annotations_final.csv"
TRAIN_SPLIT = DATA_DIR / "train_gt_mtt.tsv"
VAL_SPLIT = DATA_DIR / "val_gt_mtt.tsv"
TEST_SPLIT = DATA_DIR / "test_gt_mtt.tsv"

required_files = [ANNOTATION_FILE, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required files are missing:\n- " + "\n- ".join(missing)
        + "\nPlace the MTAT files in ./data/ and the audio files in ./data/mp3/."
    )

In [ ]:
# Replace GTZAN with MTAT: multi-label targets and 5-second audio segments at 16 kHz.
# Audio is read from disk batch by batch instead of being preloaded into RAM.

class MagnaTagATuneDataset(Dataset):
    # List of genre tags selected for this experiment.
    GENRES_TAGS = [
        'hard rock', 'clasical', 'classical', 'soft rock',
        'jazz', 'electric', 'ambient', 'new age', 'synth', 'funk', 'middle eastern',
        'medieval', 'classic', 'electronic', 'opera', 'country', 'electro', 'reggae',
        'tribal', 'dark', 'electronica', 'trance', 'chant', 'heavy metal', 'modern',
        'disco', 'industrial', 'pop', 'orchestral', 'orchestra', 'eastern', 'punk',
        'spanish', 'blues', 'indian', 'rock', 'jazzy', 'techno', 'house', 'rap', 'metal',
        'hip hop', 'baroque',
    ]

    def __init__(
        self,
        annotation_file_path,
        split_file_path,
        audio_dir="./data/mp3",
        audio_duration=5,
        sample_rate=16000,
        subsample_factor=1.0,
    ):
        super().__init__()
        if not 0 < audio_duration <= 30:
            raise ValueError("audio_duration must be between 0 and 30 seconds.")
        if not 0 < subsample_factor <= 1:
            raise ValueError("subsample_factor must be in the interval (0, 1].")

        self.audio_dir = Path(audio_dir)
        self.audio_duration = audio_duration
        self.sample_rate = sample_rate
        self.n_samples = sample_rate * audio_duration

        self.annotation_df = pd.read_csv(annotation_file_path, sep="\t")
        all_tags = [
            col for col in self.annotation_df.columns
            if col not in ["clip_id", "mp3_path"]
        ]
        self.tags_cols = [tag for tag in self.GENRES_TAGS if tag in all_tags]
        if not self.tags_cols:
            raise ValueError("The expected genre tags were not found in annotations_final.csv.")
        self.n_classes = len(self.tags_cols)

        self.data = self._filter_by_split(split_file_path)
        if self.data.empty:
            raise ValueError(f"The split {split_file_path} contains no examples with the selected tags.")

        if subsample_factor < 1.0:
            n_samples = max(1, int(len(self.data) * subsample_factor))
            self.data = self.data.sample(n=n_samples, random_state=SEED).reset_index(drop=True)

        # Keep only paths and labels in memory; audio waveforms stay on disk.
        self.audio_paths = []
        self.labels = []
        self.clip_ids = []

        for _, row in self.data.iterrows():
            clip_id = row["clip_id"]
            audio_path = self.audio_dir / row["mp3_path"]
            if not audio_path.is_file():
                raise FileNotFoundError(f"Audio file not found: {audio_path}")

            self.audio_paths.append(str(audio_path))
            self.labels.append(
                torch.tensor(
                    row[self.tags_cols].to_numpy(dtype=np.float32),
                    dtype=torch.float32,
                )
            )
            self.clip_ids.append(clip_id)

        print(
            f"{Path(split_file_path).stem}: {len(self.clip_ids)} clips; "
            f"{self.n_classes} labels. Audio will be read from disk by batch workers."
        )

    def _filter_by_split(self, split_file_path):
        split_ids = set(pd.read_csv(split_file_path, sep="\t", header=None)[0])
        split_df = self.annotation_df[
            self.annotation_df["clip_id"].isin(split_ids)
        ].copy()
        # Keep clips with at least one positive label among the selected tags.
        split_df = split_df[
            split_df[self.tags_cols].sum(axis=1) >= 1
        ].reset_index(drop=True)
        return split_df

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, idx):
        # Return metadata only; the batch collator reads and decodes the audio.
        return {
            "audio_path": self.audio_paths[idx],
            "labels": self.labels[idx],
            "clip_id": self.clip_ids[idx],
        }


def collate_audio_batch(samples):
    """Load, preprocess, and stack only the audio files in the requested batch."""
    audio_batch = []
    label_batch = []
    clip_ids = []

    for sample in samples:
        waveform, sr = torchaudio.load(sample["audio_path"])

        if sr != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(
                waveform,
                orig_freq=sr,
                new_freq=SAMPLE_RATE,
            )

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        waveform = waveform.squeeze(0).float()

        # Random 5-second crop; zero-pad clips that are shorter than the target.
        max_start = waveform.shape[0] - AUDIO_DURATION * SAMPLE_RATE
        start = (
            torch.randint(0, max_start + 1, (1,)).item()
            if max_start > 0 else 0
        )
        waveform = waveform[start:start + AUDIO_DURATION * SAMPLE_RATE]

        if waveform.shape[0] < AUDIO_DURATION * SAMPLE_RATE:
            waveform = F.pad(
                waveform,
                (0, AUDIO_DURATION * SAMPLE_RATE - waveform.shape[0]),
            )

        audio_batch.append(waveform)
        label_batch.append(sample["labels"])
        clip_ids.append(sample["clip_id"])

    return {
        "audio": torch.stack(audio_batch),
        "labels": torch.stack(label_batch),
        "clip_id": clip_ids,
    }

In [ ]:
# Create the MTAT DataLoaders with bounded batch-level audio loading.

AUDIO_DURATION = 5  # seconds
SAMPLE_RATE = 16000
BATCH_SIZE = 128
NUM_WORKERS = 0  # Load batches in the main process; avoids multiprocessing worker failures in Colab.

train_dataset = MagnaTagATuneDataset(
    ANNOTATION_FILE,
    TRAIN_SPLIT,
    audio_dir=DATA_DIR / "mp3",
    audio_duration=AUDIO_DURATION,
    sample_rate=SAMPLE_RATE,
)
val_dataset = MagnaTagATuneDataset(
    ANNOTATION_FILE,
    VAL_SPLIT,
    audio_dir=DATA_DIR / "mp3",
    audio_duration=AUDIO_DURATION,
    sample_rate=SAMPLE_RATE,
)
test_dataset = MagnaTagATuneDataset(
    ANNOTATION_FILE,
    TEST_SPLIT,
    audio_dir=DATA_DIR / "mp3",
    audio_duration=AUDIO_DURATION,
    sample_rate=SAMPLE_RATE,
)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "collate_fn": collate_audio_batch,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
}

if NUM_WORKERS > 0:
    loader_options["prefetch_factor"] = 1

train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

assert train_dataset.tags_cols == val_dataset.tags_cols == test_dataset.tags_cols
n_classes = train_dataset.n_classes

print(
    f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | "
    f"Test: {len(test_dataset)}"
)
print(f"Number of labels: {n_classes}")
print(f"Audio loading: batch-wise from {DATA_DIR / 'mp3'}")
print(f"DataLoader workers: {NUM_WORKERS}")


## 2. Define the Transformer model

### What this section does

Log-Mel spectrograms are extracted and fed into the Transformer. The initial convolution divides the spectrogram into patches and projects them into embeddings. The `TransformerEncoderLayer` blocks then process the patch sequence, and a dense head produces one logit per label.

### Changes relative to the reference

- The class `FeatureExtractor` and the in-class `Transformer` architecture are retained.
- `import math` is added because it is required to build the positional encoding already defined in the reference architecture.
- The output size is set to the number of MTAT labels instead of the 10 GTZAN classes.
- The output consists of independent logits for each label. The multi-label loss function (`BCEWithLogitsLoss`) and sigmoid are applied during training/evaluation, not inside the model.

In [ ]:
class FeatureExtractor(nn.Module):    """Log Mel-spectrogram feature extractor from the class notebook."""    def __init__(self, **kwargs):        super().__init__()        self.melspectrogram = MelSpectrogram(**kwargs)    def forward(self, x):        x = self.melspectrogram(x)        # Same logarithmic compression as in the class notebook.        return torch.log10(1 + 1000 * x)feature_extractor = FeatureExtractor(n_mels=64, n_fft=1024).to(device)# Visualize spectrograms to check feature extraction.batch = next(iter(val_loader))with torch.no_grad():    mel_specs = feature_extractor(batch["audio"].to(device, non_blocking=True)).cpu().numpy()fig, axes = plt.subplots(1, 3, figsize=(15, 4))for i, ax in enumerate(axes):    ax.imshow(mel_specs[i], aspect="auto", origin="lower")    ax.set_title(f"Example {i} — {batch['clip_id'][i]}")    ax.set_xlabel("Time")    ax.set_ylabel("Mel bin")plt.tight_layout()plt.show()print("Mel-spectrogram batch shape:", mel_specs.shape)

In [ ]:
# !!! MODIFICATION: Keep the reference architecture. The math import was added in the imports cell because positional_encoding requires it.
# !!! MODIFICATION: Set output_size to the number of MTAT labels when instantiating the model.

class Transformer(nn.Module):
    """Transformer model for audio classification.

    Args:
        kernel (tuple, optional): Kernel size for the convolutional patch embedding. Defaults to (8, 8).
        stride (tuple, optional): Stride for the convolutional patch embedding. Defaults to (8, 8).
        n_blocks (int, optional): Number of Transformer encoder blocks. Defaults to 2.
        emb_dim (int, optional): Embedding dimension. Defaults to 128.
        n_heads (int, optional): Number of attention heads. Defaults to 2.
        mlp_ratio (int, optional): Ratio of MLP hidden dimension to embedding dimension. Defaults to 2.
        output_size (int, optional): Number of output classes. Defaults to 10.
    """

    def __init__(
        self,
        kernel: tuple = (8, 8),
        stride: tuple = (8, 8),
        n_blocks: int = 2,
        emb_dim: int = 128,
        n_heads: int = 2,
        mlp_ratio: int = 2,
        output_size: int = 10,
        pos_enc_gain: float = 0.0,
        cls_token_ratio: float = 0.0,
    ):
        super().__init__()

        self.n_blocks = n_blocks
        self.emb_dim = emb_dim
        self.n_heads = n_heads
        self.mlp_ratio = mlp_ratio
        self.output_size = output_size
        self.pos_enc_gain = pos_enc_gain
        self.cls_token_ratio = cls_token_ratio

        self.norm = nn.BatchNorm2d(1)

        padding = kernel[0] // 2
        self.patching = nn.Conv2d(
            1, self.emb_dim, kernel_size=kernel, stride=stride, padding=padding
        )

        self.blocks = nn.ModuleList([
          nn.TransformerEncoderLayer(
              self.emb_dim,
              self.n_heads,
              dim_feedforward=self.emb_dim * self.mlp_ratio,
              dropout=0.1,
              norm_first=True,
          ) for _ in range(self.n_blocks)])

        self.cls = nn.Parameter(torch.randn(1, 1, self.emb_dim))

        nn.init.normal_(self.cls, std=0.02)

        self.fc1 = nn.Linear(self.emb_dim, self.emb_dim)
        self.fc2 = nn.Linear(self.emb_dim, self.output_size)

    def positional_encoding(self, seq_len):
        # !!! MODIFICATION:  Create position and div_term on the same device as the model to avoid CPU/GPU incompatibilities.
        position = torch.arange(seq_len, dtype=torch.float32, device=self.cls.device).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, self.emb_dim, 2, dtype=torch.float32, device=self.cls.device) * -(math.log(10000.0) / self.emb_dim))

        pe = torch.zeros(seq_len, self.emb_dim, device=self.cls.device)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x):
        # Add embedding dimension
        x = x.unsqueeze(1)

        # Normalize input to zero-mean unit std.
        x = self.norm(x)

        # Divide the signal into patches
        x = self.patching(x)  # [B, 1, freq, time] -> [b, E, M, T]

        B, E, M, T = x.shape

        # Generate positional encodings for frequency and time
        pe_m = self.positional_encoding(M) # (M, E)
        pe_t = self.positional_encoding(T) # (T, E)

        pe_m_reshaped = pe_m.transpose(0, 1).unsqueeze(0).unsqueeze(3) # (1, E, F, 1)
        pe_t_reshaped = pe_t.transpose(0, 1).unsqueeze(0).unsqueeze(2) # (1, E, 1, T)

        # Add positional encoding with small gain.
        x = x + self.pos_enc_gain * (pe_m_reshaped + pe_t_reshaped)

        x = x.flatten(start_dim=2)  # [B, E, F, T] -> [B, E, F * T]
        x = x.swapaxes(1, 2)  # [B, E, F * T] -> [B, F * T, E]

        # Concatenate CLS token
        cls = self.cls.repeat(B, 1, 1)
        x = torch.cat([cls, x], dim=1)

        # Transformer backbone
        for block in self.blocks:
          x = block(x)

        # Classification on top of the CLS + average of the rest of tokens
        x_cls = x[:, 0, :]
        x_mean = x[:, 1:, :].mean(dim=1)
        x = self.cls_token_ratio * x_cls + (1 - self.cls_token_ratio) * x_mean

        # Dense classification head
        x = F.layer_norm(x, (self.emb_dim,))
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

## 3. Train the model

### What this section does

Each configuration is trained with Adam and `BCEWithLogitsLoss`, which is suitable for multi-label classification. Loss, ROC-AUC, and PR-AUC are computed for train and validation at each epoch. The checkpoint with the best validation ROC-AUC is saved.

### Changes relative to the reference

The class training loop is adapted as follows:
- `CrossEntropyLoss` is replaced with `BCEWithLogitsLoss`.
- Sigmoid is used to convert logits into probabilities.
- Accuracy is replaced with macro ROC-AUC and PR-AUC.
- The training and validation history is retained.
- The best model according to validation ROC-AUC is saved.

**Additional improvement:** training metrics are recomputed in evaluation mode at the end of each epoch. This ensures they are computed using a single version of the weights, rather than mixing predictions generated before and after parameter updates within the epoch.

Labels with no positive or no negative examples in a split are excluded only from that split's metric calculations, because ROC-AUC is undefined for them. They are not removed from training or from the target vector.

In [ ]:
# !!! MODIFICATION:  Replace accuracy with macro ROC-AUC and PR-AUC, which are suitable for multi-label classification.# !!! MODIFICATION:  Add evaluation without gradients, best-checkpoint saving, early stopping, and metric history.def compute_metrics(all_probs, all_labels):    """Macro ROC-AUC and macro Average Precision for valid labels."""    positives = all_labels.sum(axis=0)    negatives = (all_labels == 0).sum(axis=0)    valid = (positives > 0) & (negatives > 0)    if not np.any(valid):        return {"roc_auc": float("nan"), "pr_auc": float("nan"), "n_valid_labels": 0}    y_true = all_labels[:, valid]    y_score = all_probs[:, valid]    return {        "roc_auc": roc_auc_score(y_true, y_score, average="macro"),        # average_precision_score computes Average Precision as a summary of the Precision-Recall curve.        "pr_auc": average_precision_score(y_true, y_score, average="macro"),        "n_valid_labels": int(valid.sum()),    }def run_inference(model, loader, loss_fn, feature_extractor, device):    model.eval()    feature_extractor.eval()    losses, probs_all, labels_all = [], [], []    with torch.no_grad():        for batch in loader:            x = batch["audio"].to(device, non_blocking=True)            y = batch["labels"].to(device, non_blocking=True)            features = feature_extractor(x)            logits = model(features)            loss = loss_fn(logits, y)            losses.append(loss.item())            probs_all.append(torch.sigmoid(logits).cpu().numpy())            labels_all.append(y.cpu().numpy())    all_probs = np.vstack(probs_all)    all_labels = np.vstack(labels_all)    return float(np.mean(losses)), all_probs, all_labelsdef train(    model,    feature_extractor,    train_loader,    val_loader,    n_epochs=50,    lr=1e-3,    device=device,    checkpoint_path="best_transformer.pth",    patience=10,):    model.to(device)    feature_extractor.to(device)    optimizer = torch.optim.Adam(model.parameters(), lr=lr)    loss_fn = nn.BCEWithLogitsLoss()    history = {        "train_loss": [], "train_roc_auc": [], "train_pr_auc": [],        "val_loss": [], "val_roc_auc": [], "val_pr_auc": [],    }    best_val_roc_auc = -float("inf")    best_epoch = 0    epochs_without_improvement = 0    for epoch in trange(n_epochs, desc="Training"):        model.train()        feature_extractor.eval()        train_losses = []        for batch in train_loader:            x = batch["audio"].to(device, non_blocking=True)            y = batch["labels"].to(device, non_blocking=True)            with torch.no_grad():                features = feature_extractor(x)            logits = model(features)            loss = loss_fn(logits, y)            optimizer.zero_grad()            loss.backward()            optimizer.step()            train_losses.append(loss.item())        # Recompute training metrics with fixed weights at the end of the epoch.        train_loss_eval, train_probs, train_labels = run_inference(            model, train_loader, loss_fn, feature_extractor, device        )        val_loss, val_probs, val_labels = run_inference(            model, val_loader, loss_fn, feature_extractor, device        )        train_metrics = compute_metrics(train_probs, train_labels)        val_metrics = compute_metrics(val_probs, val_labels)        history["train_loss"].append(train_loss_eval)        history["train_roc_auc"].append(train_metrics["roc_auc"])        history["train_pr_auc"].append(train_metrics["pr_auc"])        history["val_loss"].append(val_loss)        history["val_roc_auc"].append(val_metrics["roc_auc"])        history["val_pr_auc"].append(val_metrics["pr_auc"])        current_auc = val_metrics["roc_auc"]        print(            f"Epoch {epoch+1:03d}/{n_epochs} | "            f"train loss {train_loss_eval:.4f} ROC-AUC {train_metrics['roc_auc']:.4f} "            f"PR-AUC {train_metrics['pr_auc']:.4f} | "            f"val loss {val_loss:.4f} ROC-AUC {current_auc:.4f} "            f"PR-AUC {val_metrics['pr_auc']:.4f}"        )        if np.isfinite(current_auc) and current_auc > best_val_roc_auc:            best_val_roc_auc = current_auc            best_epoch = epoch + 1            epochs_without_improvement = 0            torch.save({                "epoch": best_epoch,                "model_state_dict": model.state_dict(),                "optimizer_state_dict": optimizer.state_dict(),                "val_loss": val_loss,                "val_roc_auc": current_auc,                "val_pr_auc": val_metrics["pr_auc"],                "history": history,            }, checkpoint_path)        else:            epochs_without_improvement += 1        # !!! MODIFICATION:  Early stopping: stop training if validation ROC-AUC does not improve for several epochs.        if epochs_without_improvement >= patience:            print(f"Early stopping: {patience} epochs without validation ROC-AUC improvement.")            break    print(f"Best validation ROC-AUC: {best_val_roc_auc:.4f} (epoch {best_epoch})")    return historydef plot_training_history(history, title):    epochs = range(1, len(history["train_loss"]) + 1)    fig, axes = plt.subplots(1, 3, figsize=(17, 4))    axes[0].plot(epochs, history["train_loss"], label="Train")    axes[0].plot(epochs, history["val_loss"], label="Validation")    axes[0].set_title("Loss")    axes[0].set_xlabel("Epoch")    axes[0].legend()    axes[0].grid(True, alpha=0.3)    axes[1].plot(epochs, history["train_roc_auc"], label="Train")    axes[1].plot(epochs, history["val_roc_auc"], label="Validation")    axes[1].set_title("ROC-AUC")    axes[1].set_xlabel("Epoch")    axes[1].legend()    axes[1].grid(True, alpha=0.3)    axes[2].plot(epochs, history["train_pr_auc"], label="Train")    axes[2].plot(epochs, history["val_pr_auc"], label="Validation")    axes[2].set_title("PR-AUC")    axes[2].set_xlabel("Epoch")    axes[2].legend()    axes[2].grid(True, alpha=0.3)    fig.suptitle(title)    plt.tight_layout()    plt.show()def load_model(model, checkpoint_path, device=device):    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)    model.load_state_dict(checkpoint["model_state_dict"])    model.to(device)    print(        f"Checkpoint: epoch {checkpoint['epoch']} | "        f"val ROC-AUC {checkpoint['val_roc_auc']:.4f} | "        f"val PR-AUC {checkpoint['val_pr_auc']:.4f}"    )    return model, checkpointdef count_parameters(model):    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Baseline

Initial lab configuration: 2 blocks, embedding dimension 64, 4 heads, non-overlapping 8×8 patches, and no positional encoding gain or CLS token weighting.

The baseline serves as the reference point for the two experiments. The checkpoint is selected based on validation ROC-AUC.

In [ ]:
# !!! MODIFICATION: Configure the model output for the number of MTAT labels and train the multi-label baseline.

N_EPOCHS = 50
LEARNING_RATE = 1e-3
PATIENCE = 10

baseline_config = {
    "kernel": (8, 8),
    "stride": (8, 8),
    "n_blocks": 2,
    "emb_dim": 64,
    "n_heads": 4,
    "mlp_ratio": 2,
    "output_size": n_classes,
    "pos_enc_gain": 0.0,
    "cls_token_ratio": 0.0,
}

baseline_model = Transformer(**baseline_config).to(device)
print(f"Baseline trainable parameters: {count_parameters(baseline_model):,}")

baseline_history = train(
    baseline_model, feature_extractor, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LEARNING_RATE, device=device,
    checkpoint_path="transformer_baseline.pth", patience=PATIENCE,
)
plot_training_history(baseline_history, "Baseline Transformer")

### Experiment 1 — Larger embedding dimension (`emb_dim=128`)

**What it aims to evaluate:** whether a higher-dimensional representation can capture richer musical features.

Only `emb_dim` changes, from 64 to 128. The number of blocks, attention heads, patches, and all other parameters remain constant. This makes it possible to study the effect of embedding dimension in isolation.

Dimension 128 is divisible by the 4 attention heads. This experiment increases the number of parameters and computational cost, so model size will also be recorded.

In [ ]:
!!! MODIFICATION: Experiment 1: change only emb_dim from 64 to 128 relative to the baseline.

embedding_config = {
    **baseline_config,
    "emb_dim": 128,  # Change relative to baseline: 64 -> 128
}
embedding_model = Transformer(**embedding_config).to(device)
print(f"Embedding experiment trainable parameters: {count_parameters(embedding_model):,}")

embedding_history = train(
    embedding_model, feature_extractor, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LEARNING_RATE, device=device,
    checkpoint_path="transformer_emb128.pth", patience=PATIENCE,
)
plot_training_history(embedding_history, "Experiment 1 — emb_dim=128")

### Experiment 2 — Greater depth (`n_blocks=4`)

**What it aims to evaluate:** whether additional Transformer blocks can build more complex representations from the spectrogram patches.

Only `n_blocks` changes, from 2 to 4. `emb_dim=64` and the rest of the baseline configuration are kept unchanged. This makes it possible to study the effect of depth without combining it with the embedding-dimension increase from Experiment 1.

In [ ]:
# !!! MODIFICATION: Experiment 2: change only n_blocks from 2 to 4 relative to the baseline.

depth_config = {
    **baseline_config,
    "n_blocks": 4,  # Change relative to baseline: 2 -> 4
}
depth_model = Transformer(**depth_config).to(device)
print(f"Depth experiment trainable parameters: {count_parameters(depth_model):,}")

depth_history = train(
    depth_model, feature_extractor, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LEARNING_RATE, device=device,
    checkpoint_path="transformer_4blocks.pth", patience=PATIENCE,
)
plot_training_history(depth_history, "Experiment 2 — n_blocks=4")

## 4. Evaluate the model

### What this section does

The best checkpoint for each configuration, selected according to validation ROC-AUC, is restored and evaluated once on the test set. `BCEWithLogitsLoss`, macro ROC-AUC, and macro Average Precision (Average Precision as a summary of the Precision-Recall curve) are computed.

The test set is not used to select hyperparameters or checkpoints. The final comparison brings together all three models to facilitate further analysis.

In [ ]:
# !!! MODIFICATION: Add final test evaluation using the best checkpoint selected exclusively on validation.

def evaluate_model(model_config, checkpoint_path, test_loader, model_name):
    model = Transformer(**model_config).to(device)
    model, checkpoint = load_model(model, checkpoint_path, device=device)

    loss_fn = nn.BCEWithLogitsLoss()
    test_loss, test_probs, test_labels = run_inference(
        model, test_loader, loss_fn, feature_extractor, device
    )
    metrics = compute_metrics(test_probs, test_labels)

    results = {
        "Model": model_name,
        "Checkpoint epoch": checkpoint["epoch"],
        "Test loss": test_loss,
        "Test ROC-AUC": metrics["roc_auc"],
        "Test PR-AUC": metrics["pr_auc"],
        "Trainable parameters": count_parameters(model),
        "Valid labels (test)": metrics["n_valid_labels"],
    }
    return results


results = []
results.append(evaluate_model(
    baseline_config, "transformer_baseline.pth", test_loader, "Baseline"
))
results.append(evaluate_model(
    embedding_config, "transformer_emb128.pth", test_loader, "Embedding 128"
))
results.append(evaluate_model(
    depth_config, "transformer_4blocks.pth", test_loader, "4 Transformer blocks"
))

results_df = pd.DataFrame(results)
display(results_df)

### Results interpretation

Complete after running the notebook:

- Which configuration achieves the best ROC-AUC on validation and test?
- Which configuration achieves the best PR-AUC?
- Does increasing `emb_dim` or `n_blocks` improve the metrics? At what cost in parameters and runtime?
- Is there a substantial gap between training and validation metrics that may indicate overfitting?
- Are the results consistent between validation and test?

**Important:** do not write conclusions before the actual results are available. ROC-AUC and PR-AUC measure different aspects and both should be discussed.